# Etapa 4 — Realce de vasos y textura

Esquema v1, etapa 4. Solo las 30 imágenes de desarrollo (`dev_ids()`).

## Preguntas

1. **¿Qué filtro de realce separa mejor los vasos del fondo**, en todos los vasos y en los finos? Se elige uno para el mapa principal (tercera componente) y otro para el verde, porque el verde con su mejor filtro es el método clásico de referencia de la pregunta principal (plan, sección 10).
2. **¿Con qué resolución se trabaja?** La etapa 2c dejó la decisión para esta etapa, con los filtros y mirando los vasos finos.
3. **Textura:** se calculan los mapas de la matriz de coocurrencia para la combinación de la etapa 5, donde se juzgan (esquema, 5.4).

## Entrada

- Mapas de la etapa 2 (`config.COLOR_MAIN_MAP`, `COLOR_REFERENCE_MAP`, `COLOR_EXTRA_MAPS`), con el mejoramiento adoptado en la etapa 3. **Este cuaderno no se corre para decidir hasta que la etapa 3 esté cerrada.**
- Cada mapa se prepara con `enhance.prepare_maps`: foto reducida si corresponde, campo de visión detectado, vasos claros (`orient`) y exterior rellenado (`fill_outside`).

## Candidatas

**Filtros**, en orden de simplicidad (de menos a más parámetros y pasos; `config.ENHANCE_FILTERS`):

0. sin filtro (el mapa de la etapa 3);
1. sombrero de copa;
2. detector de líneas;
3. filtro adaptado;
4. Gabor;
5. Frangi.

**Escalas (decisión de criterio, fijas, no se ajustan con los datos).** Salen de la etapa 1 y están en `config.py` en proporción al diámetro del campo de visión:
- anchos representativos de 2, 4, 8, 16 y 24 píxeles a resolución original (cubren los vasos finos, medianos y gruesos hasta el percentil 99 del ancho, 26 píxeles);
- σ = ancho / 2 (`sigma_for_width`): 1, 2, 4, 8 y 12 píxeles. Si al reducir la resolución un σ queda por debajo de 0,5 píxeles, se usa 0,5;
- filtro adaptado, Gabor y detector de líneas: se aplican en las cinco escalas y se toma, en cada píxel, la mayor respuesta. Filtro adaptado: largo = 4,5 σ. Gabor: elongación 2. Detector de líneas: largo = 2 × ancho;
- Frangi: las cinco σ juntas (toma el máximo internamente);
- sombrero de copa: un solo disco de radio 14 píxeles, más ancho que el vaso más grueso.

**Resolución:** original, mitad y cuarta parte (`config.ENHANCE_FACTORS`). Lo calculado a resolución reducida se agranda al tamaño original (`upscale`) antes de evaluar.

**Textura:** contraste, homogeneidad y energía (`glcm_texture`) sobre el mapa principal, con 8 niveles de gris (`quantize`, cortes por cuantiles dentro del campo detectado), distancia 1 y cuatro direcciones. Ventana de 15 píxeles a resolución original, guardada en proporción al diámetro del campo de visión. Se calcula a la resolución de trabajo que resulte de la regla 2, como los filtros.

## Medidas

- Por imagen, a resolución original, dentro del campo de visión de la base: área bajo la curva en todos los vasos (`metrics.auc`) y solo en los finos (`metrics.auc_thin`).
- Resumen: media de las imágenes. Diferencias apareadas con 2.000 remuestreos de personas, semilla 2026 (`stats.person_bootstrap_means`, `percentile_interval`).

## Reglas de decisión

1. **Filtro**, por separado para la tercera componente y para el verde, a resolución original:
   - se toman las candidatas no inferiores a la mejor **en las dos medidas** (límite inferior del intervalo al 95 % de la diferencia ≥ −0,02, `config.NONINF_MARGIN_AUC`);
   - entre ellas, la más simple según el orden de arriba. "Sin filtro" es la más simple: si es no inferior, no se filtra;
   - si ninguna candidata es no inferior en las dos medidas a la vez, se elige, entre las no inferiores en vasos finos, la de mayor área en todos los vasos. Los vasos finos tienen prioridad porque son los que más se pierden.
2. **Resolución**, con el filtro elegido para cada mapa: se adopta la más reducida que sea no inferior a la original en las dos medidas. Para el algoritmo decide la tercera componente; el verde se informa.
3. **Control:** validación cruzada por personas (`split.person_folds`, 5 grupos). En cada vuelta se repite la regla 1 con las personas de entrenamiento y se mide en las de afuera. Se informan el optimismo (entrenamiento − afuera) y si la elección es estable.
4. **Mapas complementarios:** la segunda componente y la escala de grises reciben el filtro elegido para la tercera componente, sin elección propia. El azul pasa sin filtrar: solo, su área es la del azar (0,508, etapa 2c), y un filtro no puede realzar un contraste que no tiene.
5. **Textura:** el área de cada mapa de textura se informa en los dos sentidos, solo como descripción. No se adopta ni se descarta acá: pasa a la combinación de la etapa 5, donde se juzga con la regla de 5.4 (área −0,02 y Matthews −0,03).

## Qué pasa a la etapa 5

La tercera componente y el verde con su filtro elegido, la segunda componente y la escala de grises con el filtro de la tercera, el azul sin filtrar y los tres mapas de textura, todos a la resolución elegida.

In [2]:
# En Colab, conectar Drive y usar la copia de ahí; fuera de Colab, la carpeta del repositorio.
from pathlib import Path
import os

try:
    from google.colab import drive
except ImportError:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
else:
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/TP-PSI-G8-karo_03")
os.chdir(ROOT)

Mounted at /content/drive


In [3]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import time
import numpy as np
import pandas as pd

from src.config import (COLOR_MAIN_MAP, COLOR_REFERENCE_MAP, ENHANCE_FACTORS, ENHANCE_FILTERS,
                        GLCM_LEVELS, GLCM_WINDOW_REL, NONINF_MARGIN_AUC, RESULTS_DIR, SEED)
from src.data import dev_ids, load_fov, load_image, load_split, load_vessels
from src.enhance import apply_filter, glcm_texture, prepare_maps, quantize, scale_px, upscale
from src.metrics import auc, auc_thin, thin_masks
from src.split import person_folds
from src.stats import percentile_interval, person_bootstrap_means

pd.set_option("display.precision", 4)
ids = dev_ids()
split = load_split().set_index("image")
dev = split.loc[ids]
# Corrida con la tercera componente y el verde. Después, la etapa 2d cambió
# COLOR_MAIN_MAP a "Verde"; para reproducir esta corrida se fijan acá.
COLOR_MAIN_MAP = "PC3"
MAPS = [COLOR_MAIN_MAP, COLOR_REFERENCE_MAP]        # tercera componente y verde
print(len(ids), "imágenes de desarrollo; mapas:", MAPS)

30 imágenes de desarrollo; mapas: ['PC3', 'Verde']


In [4]:
out_csv = RESULTS_DIR / "04_auc_fondo_filtros.csv"

R = pd.read_csv(out_csv)

print(len(R), "filas;", R["image"].nunique(), "imágenes")

1080 filas; 30 imágenes


## Preprocesamiento de la etapa 3
Se aplica la corrección del fondo antes de evaluar los filtros
de realce, con sigma igual a 0,02 veces el diámetro del campo
de visión automático.

In [5]:
from src.preprocess import corregir_fondo
from src.fov import fov_diameter

def preprocess(m, region):
    """Corrige el fondo con la escala utilizada en la etapa 3."""
    return corregir_fondo(
        m,
        region,
        sigma=0.02 * fov_diameter(region),
    )

STAGE3_READY = True

## Pasada 1: área de cada filtro, mapa y resolución

Para cada imagen, resolución, mapa y filtro: el filtro se aplica a la resolución de trabajo, la salida se agranda al tamaño original y se mide el área en todos los vasos y en los finos, dentro del campo de visión de la base.

- **Prueba (`TRIAL = True`):** solo 2 imágenes, en otro archivo. Sirve para ver que todo corre y cuánto tarda; no se decide nada con ella.
- **Reanudación:** el CSV se guarda después de cada imagen. Si la corrida se corta, al volver a correr la celda se saltean las imágenes ya calculadas.

In [7]:
TRIAL = not STAGE3_READY
run_ids = ids[:2] if TRIAL else ids
RESULTS_DIR.mkdir(exist_ok=True)
out_csv = RESULTS_DIR / (
    "04_auc_fondo_prueba.csv" if TRIAL
    else "04_auc_fondo_filtros.csv"
)

done = pd.read_csv(out_csv) if out_csv.exists() else pd.DataFrame()
done_ids = set(done["image"]) if len(done) else set()
rows = done.to_dict("records")

for image_id in run_ids:
    if image_id in done_ids:
        continue
    t0 = time.time()
    rgb, fov, ves = load_image(image_id), load_fov(image_id), load_vessels(image_id)
    thin = thin_masks(ves, fov)
    for factor in ENHANCE_FACTORS:
        maps, region, diameter = prepare_maps(rgb, MAPS, factor)
        for name in MAPS:
            m = preprocess(maps[name], region)
            for filt in ENHANCE_FILTERS:
                score = upscale(apply_filter(filt, m, diameter), rgb.shape[:2])
                rows.append({"image": image_id, "factor": factor, "map": name, "filter": filt,
                             "auc": auc(score, ves, fov), "auc_thin": auc_thin(score, ves, fov, masks=thin)})
        del maps
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print(f"{image_id}: {time.time() - t0:.0f} s", flush=True)

R = pd.DataFrame(rows)
print(len(R), "filas;", R["image"].nunique(), "imágenes")

12_dr: 513 s
13_dr: 497 s
14_dr: 503 s
03_g: 503 s
04_g: 495 s
05_g: 508 s
06_g: 507 s
07_g: 500 s
08_g: 509 s
11_g: 501 s
12_g: 503 s
13_g: 499 s
14_g: 498 s
1080 filas; 30 imágenes


**Medias de desarrollo** (descriptivo): área en todos los vasos y en los finos, por mapa, filtro y resolución.

In [6]:
summary = R.groupby(["map", "factor", "filter"])[["auc", "auc_thin"]].mean().unstack("factor")
summary.reindex(ENHANCE_FILTERS, level="filter")

auc                 auc_thin                
factor                         1       2       4        1       2       4
map   filter                                                             
PC3   sin filtro          0.8452  0.8647  0.8776   0.6687  0.6891  0.7005
      sombrero de copa    0.7837  0.8027  0.8477   0.6475  0.6705  0.6946
      detector de líneas  0.8575  0.8726  0.8927   0.6839  0.7053  0.7259
      filtro adaptado     0.8679  0.8681  0.8781   0.6642  0.6631  0.6782
      Gabor               0.8816  0.8867  0.8969   0.6891  0.6993  0.7143
      Frangi              0.8701  0.8846  0.8959   0.6877  0.7085  0.7177
Verde sin filtro          0.9356  0.9397  0.9393   0.8134  0.8201  0.8110
      sombrero de copa    0.9280  0.9388  0.9520   0.8410  0.8620  0.8660
      detector de líneas  0.9572  0.9598  0.9583   0.8776  0.8817  0.8693
      filtro adaptado     0.9460  0.9458  0.9448   0.8277  0.8259  0.8221
      Gabor               0.9508  0.9511  0.9496   0.8480  0.8483  0.8407
      Frangi              0.9497  0.9524  0.9478   0.8558  0.8580  0.8351

## Regla 1: filtro, a resolución original

In [7]:
def wide(R, name, factor, measure):
    # Una fila por imagen y una columna por filtro.
    sub = R[(R["map"] == name) & (R["factor"] == factor)]
    return sub.pivot(index="image", columns="filter", values=measure)[ENHANCE_FILTERS]


def noninferior(table, persons):
    # Diferencia de cada candidata con la de mayor media; límite inferior del intervalo al 95 %.
    best = table.mean().idxmax()
    diffs = table.sub(table[best], axis=0)
    lo, hi = percentile_interval(person_bootstrap_means(diffs, persons.loc[table.index]))
    return pd.DataFrame({"media": table.mean(), "diferencia": diffs.mean(),
                         "límite inferior": lo, "límite superior": hi,
                         "no inferior": np.asarray(lo) >= -NONINF_MARGIN_AUC}), best


def choose_filter(R, name, images, persons, factor=1):
    # Regla 1: la más simple entre las no inferiores en las dos medidas.
    a, best_a = noninferior(wide(R, name, factor, "auc").loc[images], persons)
    t, best_t = noninferior(wide(R, name, factor, "auc_thin").loc[images], persons)
    both = [f for f in ENHANCE_FILTERS if a.loc[f, "no inferior"] and t.loc[f, "no inferior"]]
    if both:
        choice, how = both[0], "la más simple no inferior en las dos medidas"
    else:
        thin_ok = [f for f in ENHANCE_FILTERS if t.loc[f, "no inferior"]]
        choice = a.loc[thin_ok, "media"].idxmax()
        how = "ninguna no inferior en las dos: la de mayor área entre las no inferiores en finos"
    return choice, how, a, t


persons = dev["person"]
chosen = {}
for name in MAPS:
    choice, how, a, t = choose_filter(R, name, R["image"].unique(), persons)
    chosen[name] = choice
    print(f"\n=== {name} ===  elegido: {choice}  ({how})")
    print("Todos los vasos:")
    display(a.round(4))
    print("Vasos finos:")
    display(t.round(4))


=== PC3 ===  elegido: Gabor  (la más simple no inferior en las dos medidas)
Todos los vasos:


,media,diferencia,límite inferior,límite superior,no inferior
filter,,,,,
sin filtro,0.8452,-0.0364,-0.0406,-0.0320,False
sombrero de copa,0.7837,-0.0979,-0.1127,-0.0857,False
detector de líneas,0.8575,-0.0241,-0.0314,-0.0185,False
filtro adaptado,0.8679,-0.0136,-0.0158,-0.0119,True
Gabor,0.8816,0.0000,0.0000,0.0000,True
Frangi,0.8701,-0.0115,-0.0153,-0.0078,True


Vasos finos:


,media,diferencia,límite inferior,límite superior,no inferior
filter,,,,,
sin filtro,0.6687,-0.0203,-0.0273,-0.0128,False
sombrero de copa,0.6475,-0.0416,-0.0499,-0.0333,False
detector de líneas,0.6839,-0.0051,-0.0099,-0.0005,True
filtro adaptado,0.6642,-0.0249,-0.0285,-0.0217,False
Gabor,0.6891,0.0000,0.0000,0.0000,True
Frangi,0.6877,-0.0014,-0.0049,0.0016,True



=== Verde ===  elegido: detector de líneas  (la más simple no inferior en las dos medidas)
Todos los vasos:


,media,diferencia,límite inferior,límite superior,no inferior
filter,,,,,
sin filtro,0.9356,-0.0216,-0.0254,-0.0181,False
sombrero de copa,0.9280,-0.0292,-0.0332,-0.0261,False
detector de líneas,0.9572,0.0000,0.0000,0.0000,True
filtro adaptado,0.9460,-0.0112,-0.0130,-0.0095,True
Gabor,0.9508,-0.0063,-0.0080,-0.0049,True
Frangi,0.9497,-0.0075,-0.0089,-0.0061,True


Vasos finos:


,media,diferencia,límite inferior,límite superior,no inferior
filter,,,,,
sin filtro,0.8134,-0.0642,-0.0718,-0.0566,False
sombrero de copa,0.8410,-0.0366,-0.0420,-0.0313,False
detector de líneas,0.8776,0.0000,0.0000,0.0000,True
filtro adaptado,0.8277,-0.0499,-0.0555,-0.0439,False
Gabor,0.8480,-0.0296,-0.0335,-0.0257,False
Frangi,0.8558,-0.0218,-0.0256,-0.0177,False


## Regla 2: resolución, con el filtro elegido

In [8]:
res_rows = []
for name in MAPS:
    f = chosen[name]
    for measure in ("auc", "auc_thin"):
        sub = R[(R["map"] == name) & (R["filter"] == f)]
        w = sub.pivot(index="image", columns="factor", values=measure)
        diffs = w[[k for k in ENHANCE_FACTORS if k != 1]].sub(w[1], axis=0)
        lo, hi = percentile_interval(person_bootstrap_means(diffs, persons.loc[w.index]))
        for i, k in enumerate(diffs.columns):
            res_rows.append({"mapa": name, "filtro": f, "medida": measure, "reducción": f"1/{k}",
                             "original": w[1].mean(), "reducida": w[k].mean(),
                             "diferencia": diffs[k].mean(), "límite inferior": lo[i],
                             "límite superior": hi[i], "no inferior": lo[i] >= -NONINF_MARGIN_AUC})
res = pd.DataFrame(res_rows)
display(res.round(4))

ok = res[res["mapa"] == COLOR_MAIN_MAP].groupby("reducción")["no inferior"].all()
candidates = [k for k in ENHANCE_FACTORS if k == 1 or ok.get(f"1/{k}", False)]
WORK_FACTOR = max(candidates)
print(f"Resolución de trabajo según la tercera componente: 1/{WORK_FACTOR}")

,mapa,filtro,medida,reducción,original,reducida,diferencia,límite inferior,límite superior,no inferior
0,PC3,Gabor,auc,1/2,0.8816,0.8867,0.0051,0.0045,0.0058,True
1,PC3,Gabor,auc,1/4,0.8816,0.8969,0.0153,0.0130,0.0183,True
2,PC3,Gabor,auc_thin,1/2,0.6891,0.6993,0.0103,0.0093,0.0113,True
3,PC3,Gabor,auc_thin,1/4,0.6891,0.7143,0.0252,0.0214,0.0291,True
4,Verde,detector de líneas,auc,1/2,0.9572,0.9598,0.0026,0.0020,0.0032,True
5,Verde,detector de líneas,auc,1/4,0.9572,0.9583,0.0011,-0.0002,0.0024,True
6,Verde,detector de líneas,auc_thin,1/2,0.8776,0.8817,0.0041,0.0020,0.0064,True
7,Verde,detector de líneas,auc_thin,1/4,0.8776,0.8693,-0.0083,-0.0129,-0.0031,True


Resolución de trabajo según la tercera componente: 1/4


## Regla 3: control con validación cruzada por personas

En cada vuelta se elige el filtro con las personas de entrenamiento (regla 1) y se mide en las de afuera.

In [10]:
TRIAL = not STAGE3_READY

print("Modo prueba:", TRIAL)
print("Imágenes cargadas:", R["image"].nunique())

Modo prueba: False
Imágenes cargadas: 30


In [11]:
if TRIAL:
    print("Prueba: la validación cruzada necesita las 30 imágenes; se omite.")
else:
    folds = person_folds(dev.reset_index(), k=5, seed=SEED)
    fold_of = dev["person"].map(folds)
    cv_rows = []
    for name in MAPS:
        A = wide(R, name, 1, "auc")
        T = wide(R, name, 1, "auc_thin")
        for k in range(5):
            train = [i for i in A.index if fold_of[i] != k]
            test = [i for i in A.index if fold_of[i] == k]
            choice, _, _, _ = choose_filter(R, name, train, persons)
            cv_rows.append({"mapa": name, "vuelta": k, "elegido": choice,
                            "área entrenamiento": A.loc[train, choice].mean(),
                            "área afuera": A.loc[test, choice].mean(),
                            "finos afuera": T.loc[test, choice].mean()})
    cv = pd.DataFrame(cv_rows)
    display(cv.round(4))
    for name, g in cv.groupby("mapa"):
        print(f"{name}: elegido en {g['elegido'].value_counts().to_dict()}; "
              f"optimismo {(g['área entrenamiento'] - g['área afuera']).mean():.4f}")

,mapa,vuelta,elegido,área entrenamiento,área afuera,finos afuera
0,PC3,0,Gabor,0.8859,0.8643,0.6707
1,PC3,1,Gabor,0.8833,0.8759,0.6732
2,PC3,2,Gabor,0.8821,0.8797,0.7129
3,PC3,3,Gabor,0.8789,0.8948,0.6938
4,PC3,4,Gabor,0.8779,0.8964,0.6983
5,Verde,0,detector de líneas,0.9580,0.9538,0.8711
6,Verde,1,detector de líneas,0.9597,0.9491,0.8596
7,Verde,2,detector de líneas,0.9564,0.9602,0.8945
8,Verde,3,detector de líneas,0.9568,0.9591,0.8707
9,Verde,4,detector de líneas,0.9551,0.9654,0.8940


PC3: elegido en {'Gabor': 5}; optimismo -0.0006
Verde: elegido en {'detector de líneas': 5}; optimismo -0.0003


## Textura, a la resolución de trabajo

Sobre la tercera componente. El área se informa en los dos sentidos (alto = vaso y bajo = vaso), solo como descripción: la textura se juzga dentro de la combinación de la etapa 5.

In [13]:
run_ids = ids[:2] if TRIAL else ids

print("Imágenes para textura:", len(run_ids))

Imágenes para textura: 30


In [14]:
tex_csv = RESULTS_DIR / (
    "04_auc_textura_fondo_prueba.csv" if TRIAL
    else "04_auc_textura_fondo.csv"
)
tex_rows = []
for image_id in run_ids:
    rgb, fov, ves = load_image(image_id), load_fov(image_id), load_vessels(image_id)
    thin = thin_masks(ves, fov)
    maps, region, diameter = prepare_maps(rgb, [COLOR_MAIN_MAP], WORK_FACTOR)
    m = preprocess(maps[COLOR_MAIN_MAP], region)
    window = max(3, int(round(scale_px(GLCM_WINDOW_REL, diameter))))
    tex = glcm_texture(quantize(m, GLCM_LEVELS, region), GLCM_LEVELS, window)
    for feature, t in tex.items():
        score = upscale(t, rgb.shape[:2])
        a = auc(score, ves, fov)
        tex_rows.append({"image": image_id, "textura": feature, "auc alto = vaso": a,
                         "auc bajo = vaso": 1 - a, "auc_thin alto = vaso": auc_thin(score, ves, fov, masks=thin)})
    print(image_id, end=" ", flush=True)
tex_df = pd.DataFrame(tex_rows)
tex_df.to_csv(tex_csv, index=False)
tex_df.groupby("textura").mean(numeric_only=True).round(4)

01_h 02_h 03_h 04_h 07_h 08_h 09_h 10_h 13_h 14_h 03_dr 04_dr 05_dr 06_dr 08_dr 10_dr 11_dr 12_dr 13_dr 14_dr 03_g 04_g 05_g 06_g 07_g 08_g 11_g 12_g 13_g 14_g 

,auc alto = vaso,auc bajo = vaso,auc_thin alto = vaso
textura,,,
contraste,0.4141,0.5859,0.5270
energía,0.7741,0.2259,0.4860
homogeneidad,0.7646,0.2354,0.4937


## Decisión (escrita después de correr)



Se evaluaron las 30 imágenes de desarrollo, aplicando corrección
del fondo antes de los filtros de realce.

1. **Filtro para PC3:** Gabor. Fue elegido en las cinco vueltas
   de validación cruzada por personas.

2. **Filtro para verde:** detector de líneas. Fue elegido en las
   cinco vueltas de validación cruzada por personas.

3. **Resolución de trabajo:** un cuarto del ancho y del alto.
   Cumplió el criterio de no inferioridad para todos los vasos
   y los finos en ambos mapas.

4. **Resultados a esa resolución:**
   - PC3 + Gabor: AUC todos = 0,8969; AUC finos = 0,7143.
   - Verde + detector de líneas: AUC todos = 0,9583;
     AUC finos = 0,8693.

5. **Control:** la elección de filtros fue estable en las cinco
   vueltas. Este control se realizó a resolución original;
   la resolución se eligió mediante la comparación de la Regla 2.

6. **Textura:** se calcularon contraste, homogeneidad y energía
   sobre PC3 con corrección del fondo, a la resolución de trabajo.
   Su utilidad se evaluará mediante la combinación en la etapa 5.

Pasan a la etapa 5 los filtros y la resolución elegidos, junto
con los tres mapas de textura.

In [15]:
!python -m pytest -q

........................................................................ [ 62%]
............................................                             [100%]
116 passed in 23.58s
